# M5 Full-SKU Probabilistic Forecasting & Inventory Simulation Benchmark (Google Colab Premium)

This notebook runs the two full-scale M5 experiments on **Google Colab Premium (AI Ultra)**:
1. **Part (i) — Full-SKU Multi-Section Forecasting & PIT Calibration Benchmark (`scripts/run_m5_experiments.py`)**:
   Evaluates all 5 models (`glm`, `glm_hurdle`, `catboost`, `empirical_trailing`, `empirical_seasonal`) across all 3,049 SKUs per store, with **automatic per-section checkpointing (`--resume`)** directly to Google Drive so progress is never lost if a runtime disconnects.
2. **Part (ii) — King's Formula vs. Discrete Convolution (DDLT) Inventory Simulation (`scripts/run_kings_vs_ddlt_experiment.py`)**:
   Compares King's safety-stock formula against our DDLT discrete convolution policies across target instock quantiles `80%, 85%, 90%, 95%, 99%`.

### Recommended Colab Runtime Settings
- Go to **Runtime → Change runtime type**:
  - **Hardware accelerator**: `CPU` (or `A100 GPU` if you want the 12-vCPU / 83 GB host VM; note that Numba kernels and CatBoost `MultiQuantile` run on multi-core CPU threads).
  - **Runtime shape**: Enable **High-RAM** (gives ~51 GB RAM on CPU or ~83 GB RAM on A100, eliminating the local 15 GB RAM bottleneck).
  - **CRITICAL — Background Execution**: If you plan to close your browser tab while the experiment runs, you **must** enable **Background execution** in Colab (under **Runtime → Change runtime type** or the **Connect** dropdown). Otherwise Colab disconnects and recycles the VM shortly after the tab closes, losing any in-progress section before its checkpoint is written to Drive.

In [ ]:
# 1. Verify CPU cores and available RAM on this Colab runtime
import os
import psutil

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"vCPUs available : {os.cpu_count()}")
print(f"Total RAM       : {ram_gb:.1f} GB")
if ram_gb < 20:
    print("WARNING: You are on a Standard-RAM runtime. Switch to Runtime -> Change runtime type -> High-RAM for full-SKU runs.")

In [ ]:
# 2. Mount Google Drive for persistent dataset storage and automatic per-section checkpoints
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/probabilistic_modeling_benchmark")
DRIVE_DATA_DIR = DRIVE_ROOT / "partitioned_data"
DRIVE_OUT_DIR = DRIVE_ROOT / "output"
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Persistent output & checkpoints directory: {DRIVE_OUT_DIR}")

In [ ]:
# 3. Clone repository (or upload/unzip workspace) and install package with M5 + rocket-fft extras
REPO_URL = "https://github.com/Shobhit117/DecisionCentricForecasting.git"
WORK_DIR = Path("/content/probabilistic_modeling")

if not WORK_DIR.exists():
    # Option A: If you uploaded a zip of the repo to Google Drive at DRIVE_ROOT / 'probabilistic_modeling.zip':
    zip_path = DRIVE_ROOT / "probabilistic_modeling.zip"
    if zip_path.exists():
        !unzip -q "{zip_path}" -d /content/probabilistic_modeling
    else:
        !git clone "{REPO_URL}" "{WORK_DIR}"

%cd /content/probabilistic_modeling
!pip install -q -e ".[m5,fft,dev]"

In [ ]:
# 4. Prepare or reuse partitioned M5 Parquet data stored on Google Drive
# If you already uploaded `data/m5-forecasting-accuracy/partitioned_data` to DRIVE_DATA_DIR, it is reused directly.
# Otherwise, `process_m5_dataset.py --download` downloads M5 from Zenodo and partitions it once onto Drive.
if not (DRIVE_DATA_DIR / "sales_data_CA_1.parquet").exists():
    print("Downloading and partitioning M5 dataset...")
    !python scripts/process_m5_dataset.py --download
    DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    !cp -r data/m5-forecasting-accuracy/partitioned_data/* "{DRIVE_DATA_DIR}/"
else:
    print(f"Found cached partitioned M5 data in {DRIVE_DATA_DIR}")

## Part (i): Run Full-SKU Multi-Section M5 Forecasting & PIT Benchmark

- Pass `--stores CA_1 TX_1 WI_1` (3 representative stores across all 3 states, 9,147 SKUs total) or `--stores all` (all 10 stores, 30,490 SKUs).
- The `--resume` flag writes a checkpoint to Google Drive after every `(store, section)` completes. If Colab ever disconnects, re-running the cell below resumes from the exact next section.

In [ ]:
!python scripts/run_m5_experiments.py \
    --stores CA_1 TX_1 WI_1 \
    --models all \
    --train-days 365 \
    --test-days 14 \
    --step-days 365 \
    --horizon 14 \
    --data-dir "{DRIVE_DATA_DIR}" \
    --output-dir "{DRIVE_OUT_DIR}/experiments_full_skus" \
    --resume

## Part (ii): King's Formula vs. Discrete Convolution (DDLT) Inventory Simulation

Compares King's safety-stock formula (`kings_trailing_round`, `kings_trailing_ceil`, `kings_seasonal_round`) against our discrete-convolution DDLT policies (`ddlt_glm`, `ddlt_glm_hurdle`, `ddlt_catboost`, `ddlt_empirical_trailing`, `ddlt_empirical_seasonal`) across target instock quantiles `80%, 85%, 90%, 95%, 99%`.

In [ ]:
!python scripts/run_kings_vs_ddlt_experiment.py \
    --stores CA_1 TX_1 WI_1 \
    --models all \
    --service-levels 0.80 0.85 0.90 0.95 0.99 \
    --lead-time-profiles short_concentrated right_skewed bimodal_disrupted \
    --train-days 365 \
    --test-days 28 \
    --max-sections 2 \
    --data-dir "{DRIVE_DATA_DIR}" \
    --output-dir "{DRIVE_OUT_DIR}/kings_vs_ddlt_full_skus" \
    --resume

In [ ]:
# 5. Inspect the resulting King's Formula vs. DDLT summary and Iso-Service savings tables
import pandas as pd

summary_df = pd.read_csv(DRIVE_OUT_DIR / "kings_vs_ddlt_full_skus" / "kings_vs_ddlt_summary.csv")
iso_df = pd.read_csv(DRIVE_OUT_DIR / "kings_vs_ddlt_full_skus" / "kings_vs_ddlt_iso_service.csv")
display(summary_df.head(25))
display(iso_df[iso_df["ddlt_policy"].isin(["ddlt_glm_hurdle", "ddlt_glm_hurdle_round", "ddlt_catboost"])])